Nombre Ariel Huarachi Clemente 

Nombre de la data set:Water Quality Monitoring Dataset (30-min, calidad/flujo de agua)

Url: https://www.kaggle.com/datasets/downshift/water-quality-monitoring-dataset?utm_source=chatgpt.com

importamoslibrerias

In [33]:
import os  # Manejo de rutas
import glob  # Búsqueda de archivos por patrón (para encontrar el CSV)
import warnings  # Suprimir warnings molestos
warnings.filterwarnings("ignore")  # Opcional: menos ruido

import numpy as np  # Cálculo numérico
import pandas as pd  # DataFrames y CSV
import matplotlib.pyplot as plt  # Gráficas

from sklearn.preprocessing import MinMaxScaler  # Normalización (0-1)
from sklearn.metrics import mean_absolute_error, mean_squared_error  # Métricas

import torch  # PyTorch base
import torch.nn as nn  # Módulos de red
from torch.utils.data import Dataset, DataLoader  # Dataset/Dataloader


Reproducibilidad

In [34]:

import random
SEED = 42  # Semilla global
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)


In [35]:
# Dispositivo: GPU si está disponible, si no CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cpu


In [36]:
# Carpeta donde dejaste los CSV descargados de Kaggle
DATA_DIR = "./data"

In [37]:
# Intentamos auto-encontrar un CSV dentro de ./data (ajusta si lo tienes en otra ruta)
csv_candidates = glob.glob(os.path.join(DATA_DIR, "*.csv"))
print("CSV encontrados:", csv_candidates)

CSV encontrados: ['./data\\brisbane_water_quality.csv']


In [38]:
# Elige aquí el CSV principal si hay varios (puedes cambiar el índice)
CSV_PATH = csv_candidates[0] if csv_candidates else None
print("Usando CSV:", CSV_PATH)

Usando CSV: ./data\brisbane_water_quality.csv


In [39]:
# Configuración del problema
TARGET_COL = "flow"  # <- Intención: predecir caudal; si no existe, el código buscará otra numérica
TIME_COL_CANDIDATES = ["timestamp", "time", "date", "datetime"]  # posibles nombres de columna temporal
RESAMPLE_RULE = "D"  # 'D' = diario (de 30-min a diario usando media)
WINDOW_DAYS = 60     # ventana de entrada: 60 días
HORIZON_DAYS = 30    # horizonte de salida: 30 días a predecir
BATCH_SIZE = 64      # tamaño de lote
EPOCHS = 30          # épocas máximas de entrenamiento
LR = 1e-3            # learning rate
WEIGHT_DECAY = 1e-4  # regularización L2
PATIENCE = 5         # early stopping (épocas sin mejora)

In [40]:
# 1) Cargar el CSV a DataFrame
df = pd.read_csv(CSV_PATH)
print("Shape original:", df.shape)
print("Columnas:", list(df.columns)[:20])


Shape original: (30894, 20)
Columnas: ['Timestamp', 'Record number', 'Average Water Speed', 'Average Water Direction', 'Chlorophyll', 'Chlorophyll [quality]', 'Temperature', 'Temperature [quality]', 'Dissolved Oxygen', 'Dissolved Oxygen [quality]', 'Dissolved Oxygen (%Saturation)', 'Dissolved Oxygen (%Saturation) [quality]', 'pH', 'pH [quality]', 'Salinity', 'Salinity [quality]', 'Specific Conductance', 'Specific Conductance [quality]', 'Turbidity', 'Turbidity [quality]']


In [41]:
# 2) Detectar la columna temporal si no conocemos el nombre exacto
time_col = None
for cand in TIME_COL_CANDIDATES:
    if cand in df.columns:
        time_col = cand
        break

In [42]:
# 3) Intento alternativo: detectar por contenido (fechas parseables)
if time_col is None:
    for col in df.columns:
        try:
            parsed = pd.to_datetime(df[col], errors="coerce", infer_datetime_format=True)
            if parsed.notna().mean() > 0.8:  # si más del 80% se parsea como fecha
                time_col = col
                df[col] = parsed
                break
        except Exception:
            pass
if time_col is None:
    raise ValueError("No se pudo detectar la columna de tiempo. Ajusta TIME_COL_CANDIDATES o asigna manualmente.")


In [43]:
# 4) Asegurar tipo datetime y ordenar por tiempo
df[time_col] = pd.to_datetime(df[time_col], errors="coerce", infer_datetime_format=True)
df = df.dropna(subset=[time_col]).sort_values(time_col).reset_index(drop=True)

print("Rango temporal:", df[time_col].min(), "→", df[time_col].max())


Rango temporal: 2023-08-04 23:00:00 → 2024-06-27 09:00:00


In [44]:
# 5) Fijar índice temporal y quedarnos solo con columnas numéricas
df = df.set_index(time_col)
num_df = df.select_dtypes(include=[np.number]).copy()
print("Columnas numéricas:", list(num_df.columns))


Columnas numéricas: ['Record number', 'Average Water Speed', 'Average Water Direction', 'Chlorophyll', 'Chlorophyll [quality]', 'Temperature', 'Temperature [quality]', 'Dissolved Oxygen', 'Dissolved Oxygen [quality]', 'Dissolved Oxygen (%Saturation)', 'Dissolved Oxygen (%Saturation) [quality]', 'pH', 'pH [quality]', 'Salinity', 'Salinity [quality]', 'Specific Conductance', 'Specific Conductance [quality]', 'Turbidity', 'Turbidity [quality]']


In [45]:
# 6) Confirmar/ajustar la variable objetivo
if TARGET_COL not in num_df.columns:
    print(f"TARGET_COL '{TARGET_COL}' no encontrada. Se usará la primera numérica.")
    TARGET_COL = num_df.columns[0]
print("Objetivo (y):", TARGET_COL)


TARGET_COL 'flow' no encontrada. Se usará la primera numérica.
Objetivo (y): Record number


In [46]:
# 7) Resample de 30-min a diario por media + imputación
daily = num_df.resample(RESAMPLE_RULE).mean().ffill().bfill()
print("Shape diario:", daily.shape)

Shape diario: (329, 19)


In [47]:
# 8) Split temporal 80/20 (primero) y luego 10% del train para validación → 72/8/20
N = len(daily)
split_80 = int(N * 0.80)
daily_train_full = daily.iloc[:split_80]   # 80% inicial → entrenamiento bruto
daily_test       = daily.iloc[split_80:]   # 20% final   → prueba


In [48]:
# del bloque de train, tomar 10% al final como validación (temporal)
v_size = int(len(daily_train_full) * 0.10)  # 10% del 80%  ≈ 8% del total
daily_val   = daily_train_full.iloc[-v_size:]     # últimos del bloque → validación
daily_train = daily_train_full.iloc[:-v_size]     # resto → entrenamiento

print("Split 72/8/20 →",
      "train:", len(daily_train),
      "val:", len(daily_val),
      "test:", len(daily_test))

Split 72/8/20 → train: 237 val: 26 test: 66


In [49]:
# 1) Separar features (X) y objetivo (y)
feature_cols = [c for c in daily.columns if c != TARGET_COL]  # entradas
print("Features:", feature_cols)
print("Objetivo (y):", TARGET_COL)

Features: ['Average Water Speed', 'Average Water Direction', 'Chlorophyll', 'Chlorophyll [quality]', 'Temperature', 'Temperature [quality]', 'Dissolved Oxygen', 'Dissolved Oxygen [quality]', 'Dissolved Oxygen (%Saturation)', 'Dissolved Oxygen (%Saturation) [quality]', 'pH', 'pH [quality]', 'Salinity', 'Salinity [quality]', 'Specific Conductance', 'Specific Conductance [quality]', 'Turbidity', 'Turbidity [quality]']
Objetivo (y): Record number


In [50]:
# 2) Scalers (evitar fuga: ajustar SOLO con train)
scaler_x = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train = scaler_x.fit_transform(daily_train[feature_cols]) if feature_cols else None
y_train = scaler_y.fit_transform(daily_train[[TARGET_COL]])

X_val   = scaler_x.transform(daily_val[feature_cols])   if feature_cols else None
y_val   = scaler_y.transform(daily_val[[TARGET_COL]])

X_test  = scaler_x.transform(daily_test[feature_cols])  if feature_cols else None
y_test  = scaler_y.transform(daily_test[[TARGET_COL]])

print("Shapes escalados:",
      "X_train:", None if X_train is None else X_train.shape, "y_train:", y_train.shape,
      "| X_val:", None if X_val is None else X_val.shape, "y_val:", y_val.shape,
      "| X_test:", None if X_test is None else X_test.shape, "y_test:", y_test.shape)



Shapes escalados: X_train: (237, 18) y_train: (237, 1) | X_val: (26, 18) y_val: (26, 1) | X_test: (66, 18) y_test: (66, 1)


In [51]:

# 3) Crear secuencias estilo “lab” (ventanas deslizantes)
def crear_secuencias(X_scaled, y_scaled, window=WINDOW_DAYS, horizon=HORIZON_DAYS):
    """
    X_scaled: [N, n_features] o None (si no hay features externas).
    y_scaled: [N, 1]
    Devuelve:
      X_seq -> [M, window, n_features']   (n_features' = n_features + 1 porque añadimos y como feature)
      y_seq -> [M, horizon]               (30 días por defecto)
    """
    if X_scaled is None:
        features_matrix = y_scaled  # usar y como única feature
    else:
        features_matrix = np.concatenate([X_scaled, y_scaled], axis=1)

    X_seq, y_seq = [], []
    N = len(y_scaled)
    for i in range(0, N - window - horizon + 1):
        X_seq.append(features_matrix[i:i+window])       # ventana pasado
        y_seq.append(y_scaled[i+window:i+window+horizon, 0])  # futuro
    return np.array(X_seq, dtype=np.float32), np.array(y_seq, dtype=np.float32)


In [52]:
# 4) Ventanas para cada split (72/8/20 temporal)
Xtr, ytr = crear_secuencias(X_train, y_train, WINDOW_DAYS, HORIZON_DAYS)
Xva, yva = crear_secuencias(X_val,   y_val,   WINDOW_DAYS, HORIZON_DAYS)
Xte, yte = crear_secuencias(X_test,  y_test,  WINDOW_DAYS, HORIZON_DAYS)

print("Ventanas ->",
      "Train:", Xtr.shape, ytr.shape,
      "| Val:", Xva.shape, yva.shape,
      "| Test:", Xte.shape, yte.shape)

Ventanas -> Train: (148, 60, 19) (148, 30) | Val: (0,) (0,) | Test: (0,) (0,)


In [53]:

class TimeSeriesDataset(torch.utils.data.Dataset):
    def __init__(self, X, y):
        self.X = X  
        self.y = y  
    def __len__(self):  return len(self.X)
    def __getitem__(self, idx):
        x = torch.from_numpy(self.X[idx])   # float32
        y = torch.from_numpy(self.y[idx])   # float32
        return x, y

train_ds = TimeSeriesDataset(Xtr, ytr)
val_ds   = TimeSeriesDataset(Xva, yva)
test_ds  = TimeSeriesDataset(Xte, yte)

train_dl = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_dl   = torch.utils.data.DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_dl  = torch.utils.data.DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)


In [54]:

N_FEATURES = Xtr.shape[2]  
HIDDEN     = 128
LAYERS     = 2
DROPOUT    = 0.2
HORIZON    = HORIZON_DAYS

class LSTMForecaster(nn.Module):
    def __init__(self, n_features, hidden_size, num_layers, dropout, horizon):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features, hidden_size=hidden_size,
            num_layers=num_layers, dropout=dropout, batch_first=True
        )
        self.head = nn.Linear(hidden_size, horizon)  # último hidden → 30 días

    def forward(self, x):
        out, _ = self.lstm(x)    
        last = out[:, -1, :]        # último paso
        yhat = self.head(last)      # [B, horizon]
        return yhat

def build_model():
    model = LSTMForecaster(N_FEATURES, HIDDEN, LAYERS, DROPOUT, HORIZON).to(DEVICE)
    return model

model = build_model()
print(model)


LSTMForecaster(
  (lstm): LSTM(19, 128, num_layers=2, batch_first=True, dropout=0.2)
  (head): Linear(in_features=128, out_features=30, bias=True)
)


In [55]:

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2, verbose=True
)

def train_epoch(model, loader):
    model.train()
    losses = []
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return float(np.mean(losses))

def eval_epoch(model, loader):
    model.eval()
    losses = []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            pred = model(xb)
            loss = criterion(pred, yb)
            losses.append(loss.item())
    return float(np.mean(losses))

best_val = np.inf
patience_ctr = 0
hist_train, hist_val = [], []

for epoch in range(1, EPOCHS+1):
    tr = train_epoch(model, train_dl)
    va = eval_epoch(model, val_dl)
    hist_train.append(tr); hist_val.append(va)
    scheduler.step(va)
    print(f"Época {epoch:02d}/{EPOCHS} | train_loss={tr:.6f} | val_loss={va:.6f}")

    if va < best_val - 1e-6:
        best_val = va
        patience_ctr = 0
        torch.save(model.state_dict(), "best_lstm.pt")
    else:
        patience_ctr += 1
        if patience_ctr > PATIENCE:
            print("Early stopping activado.")
            break
            print(f"\nEntrenamiento finalizado en época {epoch}")
            print(f"Mejor pérdida de validación: {best_val:.6f}")
model.load_state_dict(torch.load("best_lstm.pt", map_location=DEVICE))

# Curvas
plt.figure(figsize=(9,4))
plt.plot(hist_train, label="train_loss"); plt.plot(hist_val, label="val_loss")
plt.xlabel("Época"); plt.ylabel("MSE"); plt.title("Curvas de entrenamiento")
plt.legend(); plt.show()


Época 01/30 | train_loss=0.250998 | val_loss=nan
Época 02/30 | train_loss=0.212366 | val_loss=nan
Epoch 00003: reducing learning rate of group 0 to 5.0000e-04.
Época 03/30 | train_loss=0.124715 | val_loss=nan
Época 04/30 | train_loss=0.077397 | val_loss=nan
Época 05/30 | train_loss=0.067609 | val_loss=nan
Epoch 00006: reducing learning rate of group 0 to 2.5000e-04.
Época 06/30 | train_loss=0.055321 | val_loss=nan
Early stopping activado.


FileNotFoundError: [Errno 2] No such file or directory: 'best_lstm.pt'

In [ ]:
# ===============================================
# PARTE 7: EVALUACIÓN + PREDICCIÓN + GRÁFICOS
# ===============================================

def inverse_scale_y(arr_2d, scaler):
    M, H = arr_2d.shape
    flat = arr_2d.reshape(-1, 1)
    inv  = scaler.inverse_transform(flat)
    return inv.reshape(M, H)

model.eval()
preds_s, trues_s = [], []
with torch.no_grad():
    for xb, yb in test_dl:
        yhat = model(xb.to(DEVICE)).cpu().numpy()
        preds_s.append(yhat)
        trues_s.append(yb.numpy())

preds_s = np.concatenate(preds_s, axis=0)   # [M_test, 30]
trues_s = np.concatenate(trues_s, axis=0)   # [M_test, 30]

preds = inverse_scale_y(preds_s, scaler_y)
trues = inverse_scale_y(trues_s, scaler_y)

rmse = np.sqrt(np.mean((preds - trues)**2))
mae  = np.mean(np.abs(preds - trues))
print(f"RMSE (promedio horizonte 30d): {rmse:.4f}")
print(f"MAE  (promedio horizonte 30d): {mae:.4f}")

# comparar en la última muestra del test
k = -1
plt.figure(figsize=(10,4))
plt.plot(trues[k], label="Real (+30d)")
plt.plot(preds[k], label="Predicción (+30d)")
plt.title(f"Predicción futura (30 días) — y: {TARGET_COL}")
plt.xlabel("Días +t"); plt.ylabel(TARGET_COL)
plt.legend(); plt.show()
